In [3]:
from konlpy.tag import Okt
from collections import Counter
import re

okt = Okt()

# 불용어 정의
stopwords = set(["그", "저", "것", "이", "저는", "제가", "근데", "좀", "그냥", "정말", "되게", "음", "뭐"])

# 추임새로 자주 쓰이는 형태소 리스트 (필요하면 추가 가능)
fillers = {"어", "음", "아", "그", "저기", "뭐", "응", "흠"}

# 텍스트 정제
def clean_text(text):
    text = re.sub(r"[^가-힣\s]", "", text)
    return text.strip()

# 자주 쓰는 단어 추출
def extract_keywords(texts, min_len=2, top_k=20):
    counter = Counter()
    for text in texts:
        text = clean_text(text)
        nouns = okt.nouns(text)
        nouns = [n for n in nouns if len(n) >= min_len and n not in stopwords]
        counter.update(nouns)
    return counter.most_common(top_k)

# 문장 끝 표현 추출
def extract_sentence_endings(texts):
    endings = []
    for text in texts:
        sentences = re.split(r'[.!?]', text)
        for sentence in sentences:
            sentence = sentence.strip()
            if sentence:
                morphs = okt.morphs(sentence)
                if morphs:
                    endings.append(morphs[-1])
    return Counter(endings).most_common(10)

# 추임새 추출 함수
def extract_fillers(texts):
    filler_counter = Counter()
    for text in texts:
        morphs = okt.morphs(text)
        filler_counter.update([m for m in morphs if m in fillers])
    return filler_counter.most_common(10)

# 종합 분석 함수
def analyze_user_style(texts):
    result = {}
    result["keywords"] = extract_keywords(texts)
    result["endings"] = extract_sentence_endings(texts)
    result["fillers"] = extract_fillers(texts)
    return result

# 분석 결과 출력
def print_analysis(result):
    print("🧠 [말버릇 분석 결과]\n")

    print("\n 자주 사용하는 단어 (상위 5개):")
    if result["keywords"]:
        for word, freq in result["keywords"][:5]:
            print(f"  - {word}: {freq}회")
    else:
        print("  ❌ 없음")

    print("\n 문장 끝 표현 (상위 5개):")
    if result["endings"]:
        for ending, count in result["endings"][:5]:
            print(f"  - '~{ending}': {count}회")
    else:
        print("  ❌ 없음")
    
    print("\n 자주 사용하는 추임새 (상위 5개):")
    if result["fillers"]:
        for filler, count in result["fillers"][:5]:
            print(f"  - '{filler}': {count}회")
    else:
        print("  ❌ 없음")

    print("\n✅ 분석 완료\n")

# === 메인 로직 ===
if __name__ == "__main__":
    # 분석할 텍스트 예시
    stt_result_text = "어 최근에 이제 디자인 업무에서도 글로벌 인재를 선호하고 있습니다. 아무래도 이제 소비자의 계층이 우리나라 뿐만이 아니라 어 여러 세계의 글로발한 나라에 초점이 맞춰져 있고 또 그리고 세계 시장으로 진출하려는 기업들이 늘어나다 보니까 어 해외 문화나 그런 정서를 받아들여서 디자인을 하는 추세인 것 같습니다. 그리고 수출이 이뤄지게 되면 그게 국내에서만 사용되는 디자인이 아니라 해외에서도 모두 사용이 되기 때문에 그런 것들을 개발할 때 좀 글로벌한 시각을 보고 연구할 수 있는 그런 디자이너들이 많이 선호가 되는 거 같습니다. 어 저도 그래서 그 디자인을 할 때 국내 정서와 문화에만 가치 기반을 두지 않고요. 이런 것들이 전세계적으로 좀 일반적으로 공통되고 이해할 수 있는 쪽으로 많이 관심을 가지려고 합니다. 어 실제로 제가 해외 디자인 프로젝트에도 참여해 본 적이 있는데요. 어 우리나라와 좀 다른 문화와 정서를 가지고 있어서 그런지 또 제가 배울 점이 굉장히 많았다고 생각을 합니다. 저도 이제 글로벌 인 한 그런 디자인을 만들기 위해서 여러 가지 배우고 느끼면서 접목할려고 하고 있고요. 그리고 가장 기본은 문화의 다양성을 존중하면서도 좀 심층 깊은 그런 아이디어를 내기 위해서 노력을 하고 있는 상황입니다."

    if not stt_result_text.strip():
        print("⛔ 분석할 텍스트가 없습니다.")
    else:
        user_texts = [stt_result_text]
        analysis = analyze_user_style(user_texts)
        print_analysis(analysis)


🧠 [말버릇 분석 결과]


 자주 사용하는 단어 (상위 5개):
  - 디자인: 6회
  - 문화: 4회
  - 이제: 3회
  - 글로벌: 3회
  - 해외: 3회

 문장 끝 표현 (상위 5개):
  - '~같습니다': 2회
  - '~합니다': 2회
  - '~있습니다': 1회
  - '~않고요': 1회
  - '~있는데요': 1회

 자주 사용하는 추임새 (상위 5개):
  - '어': 6회
  - '그': 1회

✅ 분석 완료

